# Homework 10 - ST - 554 Big data analysis

## Author: Yefrid Cordoba

### Part 1

First we need to Setup a data stream using the `"rate"` format.\
Also we import the required functions to complete this part (`sqrt()`, `pmod()`)

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import pmod, sqrt
spark = SparkSession.builder.getOrCreate()
rateDF = spark.readStream.format("rate").load()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/20 19:09:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Now we create a query that is going to take the value generated from the rate streaming data and calculate the squared root and the mod 4, and put those values in two new columns in the dataframe that is being generated.

In [12]:
rateDF = rateDF.withColumn("sqrt", sqrt(rateDF.value))\
                .withColumn("mod4", pmod(rateDF.value, 4))

We check that the dataframe that we get as output after start the stream is the expected, including the two new rows generated.

In [13]:
rateDF

DataFrame[timestamp: timestamp, value: bigint, sqrt: double, mod4: bigint]

As the expected dataframe is correct, we start the streaming data, we let it run for about 30 seconds and then stop it.

In [16]:
new_stm = rateDF\
            .writeStream\
            .outputMode("append")\
            .format("memory")\
            .queryName("stm_data")\
            .start()

26/04/20 16:37:49 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-6b63e08e-bfd5-4687-a073-21febf18a49b. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/20 16:37:49 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


After the selected time for run, we execute the next code chunck to stop the values generation. 

In [17]:
new_stm.stop()

26/04/20 16:38:34 WARN DAGScheduler: Failed to cancel job group c363c6a3-71bf-490d-8b34-42f41f0d4a5f. Cannot find active jobs for it.
26/04/20 16:38:34 WARN DAGScheduler: Failed to cancel job group c363c6a3-71bf-490d-8b34-42f41f0d4a5f. Cannot find active jobs for it.


And we can check the data that was stored in the memory during the execution of the code.

In [21]:
spark.sql("SELECT * FROM stm_data").show()

+--------------------+-----+------------------+----+
|           timestamp|value|              sqrt|mod4|
+--------------------+-----+------------------+----+
|2026-04-20 16:37:...|    0|               0.0|   0|
|2026-04-20 16:37:...|    1|               1.0|   1|
|2026-04-20 16:37:...|    2|1.4142135623730951|   2|
|2026-04-20 16:37:...|    3|1.7320508075688772|   3|
|2026-04-20 16:37:...|    4|               2.0|   0|
|2026-04-20 16:37:...|    5|  2.23606797749979|   1|
|2026-04-20 16:37:...|    6| 2.449489742783178|   2|
|2026-04-20 16:37:...|    7|2.6457513110645907|   3|
|2026-04-20 16:37:...|    8|2.8284271247461903|   0|
|2026-04-20 16:37:...|    9|               3.0|   1|
|2026-04-20 16:38:...|   10|3.1622776601683795|   2|
|2026-04-20 16:38:...|   11|   3.3166247903554|   3|
|2026-04-20 16:38:...|   12|3.4641016151377544|   0|
|2026-04-20 16:38:...|   13| 3.605551275463989|   1|
|2026-04-20 16:38:...|   14|3.7416573867739413|   2|
|2026-04-20 16:38:...|   15| 3.872983346207417

We can see that everything worked fine during the streaming of the data for the new columns that were created.

### Part 2

In [3]:
import pandas as pd
from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, VectorAssembler
spark = SparkSession.builder.getOrCreate()

First we are going to import the `bikeDetails_for_fit.csv` as a pandas dataframe, and then turn it into a SQL dataframe and use its schema for the readStream process.

In [18]:
bike_stm = pd.read_csv('bikeDetails_for_fit.csv')

In [21]:
bike_stm = spark.createDataFrame(bike_stm)

In [22]:
bike_stm.schema

StructType([StructField('name', StringType(), True), StructField('selling_price', LongType(), True), StructField('year', LongType(), True), StructField('seller_type', StringType(), True), StructField('owner', StringType(), True), StructField('km_driven', LongType(), True), StructField('ex_showroom_price', DoubleType(), True)])

In [23]:
my_schema = bike_stm.schema

We are going to create a sql transformer to select and compute some calculations just using and keeping the columns that are going to be used.

In [24]:
d_trans = """
            SELECT log(selling_price) as label, year, log(km_driven) as log_km_driven,
            CASE WHEN owner = '1st owner' THEN 1 ELSE 0 END AS one_owner
            FROM __THIS__

            """

In [25]:
sql_trans = SQLTransformer(statement = d_trans)

Also we are going to combine the features in one column as a vector in preparation to use this data for fitting models in MLlib

In [26]:
feature_cols = ['year', 'log_km_driven', 'one_owner']
f_assem = VectorAssembler(inputCols = feature_cols,\
                         outputCol = 'features')


With all the transformations ready we can set up the pipeline for the data that is going to ve streamed.

In [27]:
bike_pipeline = Pipeline(stages = [sql_trans, f_assem])

And fit the model using the initial data.

In [28]:
bike_d = bike_pipeline.fit(bike_stm)

Next step is to set up the read stream system for a folder called `bike_data`, and we are going to use the schema from the first read file.

In [29]:
new_bike = spark.readStream.schema(my_schema).csv("bike_data", header = True)

We write how we want the data to be transformed, that is usig the transform method from the previously fitted `pipelinemodel` to the fit file.

In [30]:
trans = bike_d.transform(new_bike)

In [31]:
trans

DataFrame[label: double, year: bigint, log_km_driven: double, one_owner: int, features: vector]

Now we set up the writing query to be printed in the console using the append mode and start, followed by the sequential addition of the bike files to the `bike_data` folder.

In [32]:
query = trans.writeStream.outputMode("append").format("console").start()

26/04/20 19:39:14 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-504fdbdd-3c7f-41f3-9f24-9abb994e1904. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/20 19:39:14 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+------------------+----+------------------+---------+--------------------+
|             label|year|     log_km_driven|one_owner|            features|
+------------------+----+------------------+---------+--------------------+
| 8.987196820661973|2003|10.887436932884098|        1|[2003.0,10.887436...|
|11.156250521031495|2018| 9.615805480084347|        1|[2018.0,9.6158054...|
|10.819778284410283|2016| 8.987196820661973|        1|[2016.0,8.9871968...|
| 10.46310334047155|2015|10.582738627903963|        1|[2015.0,10.582738...|
| 9.903487552536127|2006|11.225243392518447|        1|[2006.0,11.225243...|
|10.819778284410283|2012|10.239959789157341|        1|[2012.0,10.239959...|
| 10.51867319162636|2008| 11.03488966402723|        1|[2008.0,11.034889...|
|11.141861783579396|2018| 9.392661928770137|        1|[2018.0,9.3926619...|
|10.239959789157341|2012| 10.81975828421028|        1|[2012.0,10.81

And we stop the query, so we stop looking for new files in the especified folder.

In [33]:
query.stop()

26/04/20 19:40:03 WARN DAGScheduler: Failed to cancel job group dc91d96f-9304-4a29-863a-308480877e7d. Cannot find active jobs for it.
26/04/20 19:40:03 WARN DAGScheduler: Failed to cancel job group dc91d96f-9304-4a29-863a-308480877e7d. Cannot find active jobs for it.


Now it is complete and we can see that for each of the files, a new batch of data was transformed using the previous pipeline.